In this notebook, we compute time-varying bias and random error parameters per month, so that the basin-scale prior mean `mPriorB` and 
and the basin-scale prior variance `vPriorB` matches the basin-scale posterior mean `mPostB` and basin-scale posterior `vPostB` in each month. 
In other words, in each month we set all parameters, including: wE and rE (and wP and rP) in the prior such that the innovation terms in the Kalman eq. (28-29) in the paper become zero. 
The results are timevarying parameters that we can use directly in the Kalman eq to update the grid-scale prior mean and variance, for each of $P$ and $E$ variables, to get the posterior gridded data sets.

In [1]:
#useful imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
import glob
from scipy.stats import norm#the normal distribution
from matplotlib.ticker import MultipleLocator, AutoMinorLocator
from matplotlib.font_manager import FontProperties
import seaborn as sns
from collections import OrderedDict
import datetime
import matplotlib.dates as mdates
from scipy.stats import norm

In [2]:
# Load data (priors as txt files)
PObs1 = np.loadtxt("minP.txt")
PObs2 = np.loadtxt("maxP.txt")
PStd = np.loadtxt("gpmErrorC.txt")
EStd = np.loadtxt("ErrorE.txt")
EObs1 = np.loadtxt("minET.txt")
EObs2 = np.loadtxt("maxET.txt")
QObs = np.loadtxt("runoff.txt")
IObs = np.loadtxt("waterImports.txt")
SObs = np.loadtxt("graceMAS.txt")
# Create df to hold priors
priorDict = { # prior dictionary
    "Pmin": PObs1,
    "Pmax": PObs2,
    "PStd": PStd,
    "EStd": EStd,
    "Emin": EObs1,
    "Emax": EObs2,
    "QObs": QObs,
    "IObs": IObs,
    "SObs": SObs}

dfPriors = pd.DataFrame(priorDict)

print(dfPriors.head())


        Pmin       Pmax      PStd       EStd        Emin        Emax  \
0  35.977177  80.255710  8.483780   5.950775   24.020277   54.842422   
1  27.074856  40.483047  2.561446   4.897061   66.480423   91.766724   
2  10.376050  20.235334  1.885363   4.912141   90.569145  116.745247   
3   2.673420   5.796332  0.610593   7.610889  100.040588  139.785294   
4  10.717604  37.658432  5.280222  12.300309   78.093605  141.930924   

       QObs       IObs     SObs  
0  7.930659  12.619505  263.994  
1  9.507489  13.899295  308.181  
2  8.513443   0.423866  342.195  
3  4.977012  15.943469  315.336  
4  8.915984  15.704838  278.912  


In [3]:
posteriors_files = [(file, open(file, 'r').read()) for file in glob.glob('*.out') if file != 'resultsPar.out']

# parse posteriors files 
posteriors = {}
for file, data in posteriors_files:
    filename = file.split('.')[0]
    parsed_data = [tuple(map(float, entry.strip('Gaussian()').replace('(', '').replace(')', '').split(','))) for entry in data.split('\n') if entry]
    mean = [entry[0] for entry in parsed_data]
    variance = [entry[1] for entry in parsed_data]
    posteriors['mPost'+filename.split('results')[1]] = mean
    posteriors['vPost'+filename.split('results')[1] ] = variance

# Create df to hold posteriors
dfPosteriors = pd.DataFrame(posteriors)

print(dfPosteriors.head())

   mPostE  vPostE  mPostS  vPostS  mPostQ  vPostQ  mPostP   vPostP   mPostI  \
0   40.40   35.85   408.7   599.1   7.925  0.6319  54.420  78.8900  12.7200   
1   73.37   27.96   372.8   567.9   9.477  0.9076  32.660   7.1880  14.3100   
2   94.59   33.09   284.7   563.4   8.479  0.7279  14.490   3.8930   0.4244   
3  110.50   65.90   189.7   591.2   4.967  0.2492   3.943   0.4058  16.5700   
4  105.60  147.40   113.5   637.2   8.896  0.7982  22.160  29.8200  16.0900   

     vPostI  
0  10.04000  
1  12.15000  
2   0.01133  
3  15.92000  
4  15.46000  


In [4]:
dfPosteriors["wP"]=(dfPosteriors["mPostP"]-dfPriors["Pmin"])/(dfPriors["Pmax"]-dfPriors["Pmin"])

In [5]:
fE=0.8702
dfPosteriors["wE"]=(dfPosteriors["mPostE"]-fE*(dfPriors["Emin"]))/(fE*(dfPriors["Emax"]-dfPriors["Emin"]))

In [6]:
# 0.25*avg(SRS)
vPriorE=(dfPriors["EStd"])**2
vPriorP=(dfPriors["PStd"])**2

In [7]:
dfPosteriors["rE"]=np.sqrt(dfPosteriors["vPostE"]/vPriorE)
dfPosteriors["rP"]=np.sqrt(dfPosteriors["vPostP"]/vPriorP)

In [8]:
dfPosteriors

,mPostE,vPostE,mPostS,vPostS,mPostQ,vPostQ,mPostP,vPostP,mPostI,vPostI,wP,wE,rE,rP
0,40.40,35.85,408.7,599.1,7.925,0.6319,54.4200,78.890000,12.7200,10.04000,0.416518,0.726939,1.006169,1.046939
1,73.37,27.96,372.8,567.9,9.477,0.9076,32.6600,7.188000,14.3100,12.15000,0.416547,0.705264,1.079775,1.046692
2,94.59,33.09,284.7,563.4,8.479,0.7279,14.4900,3.893000,0.4244,0.01133,0.417267,0.692617,1.171056,1.046519
3,110.50,65.90,189.7,591.2,4.967,0.2492,3.9430,0.405800,16.5700,15.92000,0.406537,0.677869,1.066614,1.043289
4,105.60,147.40,113.5,637.2,8.896,0.7982,22.1600,29.820000,16.0900,15.46000,0.424723,0.677626,0.987035,1.034193
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
235,127.60,543.70,-1075.0,643.4,6.364,0.4077,127.2000,25.130000,30.4600,55.29000,0.412429,0.679048,0.859714,1.053417
236,85.12,190.50,-980.6,626.4,10.620,1.1300,159.4000,39.310000,30.5400,59.41000,0.395165,0.762106,0.949168,1.049253
237,60.63,269.20,-997.6,652.9,7.321,0.5398,46.1600,6.689000,4.7980,1.43700,0.409639,0.678781,0.938212,1.047115
238,49.67,195.80,-1041.0,736.3,3.482,0.1223,1.2270,0.036730,8.6270,4.65100,0.402192,0.717528,0.968244,1.047074


In [9]:
#means
dfPosteriors["mPriorP"]=dfPriors["Pmin"]+dfPosteriors["wP"]*(dfPriors["Pmax"]-dfPriors["Pmin"])
dfPosteriors["mPriorE"]=fE*(dfPriors["Emin"]+dfPosteriors["wE"]*(dfPriors["Emax"]-dfPriors["Emin"]))
dfPosteriors["mdiffP"]=dfPosteriors["mPostP"]-dfPosteriors["mPriorP"]
dfPosteriors["mdiffE"]=dfPosteriors["mPostE"]-dfPosteriors["mPriorE"]
#variances
dfPosteriors["vPriorP"]=(dfPosteriors["rP"])**2 * vPriorP
dfPosteriors["vPriorE"]=(dfPosteriors["rE"])**2 * vPriorE
dfPosteriors["vdiffP"]=dfPosteriors["vPostP"]-dfPosteriors["vPriorP"]
dfPosteriors["vdiffE"]=dfPosteriors["vPostE"]-dfPosteriors["vPriorE"]
#P and E
dfPosteriors["vPriorPE"]=dfPosteriors["vPriorP"] * dfPosteriors["vPriorE"]

In [10]:
dfPosteriors.to_excel("dfPosteriors.xlsx")

In [11]:
dfPosteriors.to_csv("dfPosteriors.csv")

In [12]:
dfPosteriors["vPriorE"]

0       35.85
1       27.96
2       33.09
3       65.90
4      147.40
        ...  
235    543.70
236    190.50
237    269.20
238    195.80
239     87.17
Name: vPriorE, Length: 240, dtype: float64